Looking at the effect of landmark number and distribution on registration quality

In [1]:
import napari
import spatialdata 
import sopa
import napari_spatialdata 
import spatialdata_plot 

c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\spatialdata\_core\query\relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.memb

In [44]:
#Create spatial objects containing the images you want to align

#eventually need these to contain every xen/pheno channel. Will start by registering xenDapi and one pheno channel to get the affine, as trying all of them may cause memory issues. 


xen_image = sopa.io.ome_tif("D:/Dom/Fibrosis project/4th year data - update location/pheno_xen_fullres_images/morphology_focus_scaledXY_dapi.ome.tif", as_image = False) #ensure 'as.image = False', otherwise you create an image not a spatialdata object --> could do true, then create a dictionary with all images, then add them to a single spatial data object in future. 
pheno_image = sopa.io.ome_tif("D:/Dom/Fibrosis project/4th year data - update location/pheno_xen_fullres_images/pheno_C12.ome.tif", as_image = False) #12th channel


In [ ]:
from napari_spatialdata import Interactive
Interactive([xen_image, pheno_image]).run()

2025-11-21 16:31:14.430 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.


2025-11-21 16:31:27.499 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:31:27.500 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:31:32.355 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:31:32.359 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:31:32.359 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:32:05.799 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:32:05.803 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:34:08.915 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:34:08.920 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:34:11.506 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updati

INFO: Selected 11 points in this slice, use Shift-A to select all points on the layer. (11 selected)
INFO: Deselected all points in this slice, use Shift-A to deselect all points on the layer. (0 selected)


In [17]:
import pandas as pd
import numpy as np
from spatialdata.models import PointsModel
from spatialdata.transformations import Identity

# Read your CSV file
landmarks_df = pd.read_csv('D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/systematising landmarks/Landmark results/10_xen_space.csv')
pheno_landmarks_df = pd.read_csv('D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/systematising landmarks/Landmark results/10_pheno_space.csv')

# Extract coordinates (axis-1 = x, axis-2 = y)
# SpatialData expects coordinates in (y, x) order for 2D points
coords = landmarks_df[['axis-2', 'axis-1']].values
pheno_coords = pheno_landmarks_df[['axis-2', 'axis-1']].values

# Create a DataFrame in the format expected by PointsModel
# PointsModel expects columns: ['x', 'y'] or ['z', 'y', 'x'] for 3D
points_df = pd.DataFrame({
    'y': landmarks_df['axis-1'].values,
    'x': landmarks_df['axis-2'].values
})

pheno_points_df = pd.DataFrame({
    'y': pheno_landmarks_df['axis-1'].values,
    'x': pheno_landmarks_df['axis-2'].values
})

# Create the points element with transformation to global coordinate system
points_element = PointsModel.parse(
    points_df,
    transformations={"global": Identity()}
)

pheno_points_element = PointsModel.parse(
    pheno_points_df,
    transformations={"global": Identity()}
)

# Add to your xen_image SpatialData object
xen_image.points['xen_landmarks'] = points_element
pheno_image.points['pheno_landmarks'] = pheno_points_element 

from spatialdata.transformations import (
    align_elements_using_landmarks,
    get_transformation_between_landmarks,
)

reference_name = list(xen_image.images.keys())[0]
affine = align_elements_using_landmarks(
    references_coords=xen_image["xen_landmarks"], 
    moving_coords=pheno_image["pheno_landmarks"],
    reference_element=xen_image[reference_name],
    moving_element=pheno_image["pheno_C12"],
    reference_coordinate_system="global",
    moving_coordinate_system="global",
    new_coordinate_system="aligned",
)
affine

save_xen = xen_image
xen_path = "D:/Dom/Fibrosis project/4th year data - update location/xen_pheno_registration/systematising landmarks/aligned objects/" + "10_space_xen.zarr"
xen_image.write(xen_path)
xen_image.write_transformations()

save_pheno = pheno_image
pheno_path = "D:/Dom/Fibrosis project/4th year data - update location/xen_pheno_registration/systematising landmarks/aligned objects/" + "10_space_pheno.zarr"
pheno_image.write(pheno_path)
pheno_image.write_transformations()

c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\spatialdata\_core\_elements.py:118: UserWarning: Key `xen_landmarks` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\spatialdata\_core\_elements.py:118: UserWarning: Key `pheno_landmarks` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


INFO     The Zarr backing store has been changed from D:\Dom\Fibrosis project\4th year data - update               
         location\xen_pheno_registration\systematising landmarks\aligned objects\10_mono_xen.zarr the new file     
         path: D:\Dom\Fibrosis project\4th year data - update location\xen_pheno_registration\systematising        
         landmarks\aligned objects\10_space_xen.zarr                                                               
INFO     The Zarr backing store has been changed from D:\Dom\Fibrosis project\4th year data - update               
         location\xen_pheno_registration\systematising landmarks\aligned objects\10_mono_pheno.zarr the new file   
         path: D:\Dom\Fibrosis project\4th year data - update location\xen_pheno_registration\systematising        
         landmarks\aligned objects\10_space_pheno.zarr                                                             


In [41]:
xen_sd = spatialdata.read_zarr("D:/Dom/Fibrosis project/4th year data - update location/xen_pheno_registration/systematising landmarks/aligned objects/10_bi_xen.zarr")
pheno_sd = spatialdata.read_zarr("D:/Dom/Fibrosis project/4th year data - update location/xen_pheno_registration/systematising landmarks/aligned objects/10_bi_pheno.zarr")

version mismatch: detected: RasterFormatV02, requested: FormatV04


c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compres

In [42]:

xen_sd.points['xen_landmarks'] = xen_sd.transform_element_to_coordinate_system('xen_landmarks', 'aligned')

pheno_sd.points['pheno_landmarks'] = pheno_sd.transform_element_to_coordinate_system('pheno_landmarks', 'aligned')




c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\spatialdata\_core\_elements.py:118: UserWarning: Key `xen_landmarks` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
c:\python projects\pheno_xen_registration\.venv\Lib\site-packages\spatialdata\_core\_elements.py:118: UserWarning: Key `pheno_landmarks` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


In [ ]:
from napari_spatialdata import Interactive
Interactive([xen_sd, pheno_sd]).run()


2025-11-21 16:03:11.439 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.


2025-11-21 16:03:19.757 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:19.757 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:21.298 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:21.301 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:21.302 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:37.677 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:37.681 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.


INFO: Selected 10 points across all slices, including 0 points not currently visible. (10)


2025-11-21 16:03:50.111 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:50.114 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:03:50.114 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:04:11.854 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:04:11.857 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:04:11.858 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:04:25.808 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:04:25.810 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-11-21 16:04:25.811 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
